In [1]:
# This code is part of QCMet.
# 
# (C) Copyright 2024 National Physical Laboratory and National Quantum Computing Centre 
# 
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
# 
#      http://www.apache.org/licenses/LICENSE-2.0
# 
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License


## Quantum approximate optimization algorithm (QAOA) metric
### Q-score

In [2]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit.circuit import Parameter
from scipy.optimize import minimize
import networkx as nx
from tqdm.autonotebook import tqdm

/tmp/ipykernel_1049848/2819876981.py:8: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [3]:
import sys
import pathlib
import os
sys.path.insert(0, str(pathlib.Path(os.path.abspath('')).parent.parent.parent.resolve()))

from _helpers.circuit_submitter import CircuitSubmitter

## Define problem parameters

In [4]:
min_n = 4
max_n = 8
depth = 2

In [8]:
class QScoreBenchmark():
    def __init__(self, min_n, max_n, depth, submitter):
        self.min_n = min_n
        self.max_n = max_n
        if (self.max_n < self.min_n):
            raise ValueError("min_n must be smaller than or equal to max_n")
        self.depth = depth
        # self.backend = backend
        self.submitter = submitter
        self.graphs_per_n = 100   # For Q-Score this is chosen to be 100 
        self.shots = 2048       # For Q-Score this is chosen to be 2048
    
    def generate_graph(self, n_vertices):
        """Generate a random graph with each possible edge having a probability of 1/2 to be included"""
        graph = nx.Graph()

        vertices = list(range(n_vertices))
        graph.add_nodes_from(vertices)

        for v1 in vertices:
            for v2 in vertices:
                if v1 != v2 and np.random.rand() < 0.5:
                    graph.add_edge(v1, v2)
        
        self.graph = graph
        return graph
    
    def create_qaoa_circuit(self, params: list[Parameter]):
        qc = QuantumCircuit(QuantumRegister(self.n), ClassicalRegister(self.n))
        params = np.array(params)
        params = params.reshape(self.depth , (self.n + len(self.graph.edges)))
        betas = params[:, :self.n]
        gammas = params[:, self.n:]
        
        # Create layers of circuits
        for i in range(depth):
            for j, (v1, v2) in enumerate(self.graph.edges):
                qc.rzz(2 * gammas[i][j], v1, v2)
            for k in range(self.n):
                qc.rx(2 * betas[i][k], i)

        qc.measure_all()

        return qc
    
    def cost_func(self, parameters):
        qc = self.qc.assign_parameters(parameters=parameters)
        shots = 1000
        tasks = self.submitter.submit_circuits(shots=shots, skip_asking=True, qasm_strs=[qc.qasm()], print_summary = False)
        counts = self.submitter.convert_counts_to_qiskit(
            self.submitter.retrieve_counts([task.id for task in tasks], wait=True, print_timestamp_when_done = False)[0])
        # counts = self.backend.run(qc, shots=self.shots).result().get_counts()
        total_cost = 0
        for bit_string, count in counts.items():
            expectation = 0
            for i, j in self.graph.edges():
                if bit_string[i] != bit_string[j]:
                    expectation -= 1
            expectation -= len(self.graph.edges) / 2
            total_cost += expectation * count
        return total_cost / self.shots

    def run(self):
        max_n_passed = None
        for n in range(self.min_n, self.max_n+1):
            self.n = n
            beta_star = 0

            for _ in tqdm(range(self.graphs_per_n)):
                self.graph = self.generate_graph(n)
                no_params = self.depth * (n + len(self.graph.edges))
                self.qc = self.create_qaoa_circuit([Parameter(str(i)) for i in range(no_params)])

                res = minimize(self.cost_func, x0=np.ones(no_params), method="COBYLA")
                # The cost needs to multiply by -1 to compute beta_star
                beta_star += (-res.fun - n**2 / 8) / (0.178 * n**(3/2))

            beta_star /= self.graphs_per_n
            passed = beta_star > 0.2

            print(fr"Q-Score benchmark with n={n} gives $\beta^*$={beta_star}{' > 0.2. Passed!' if passed else ' < 0.2. Failed!'}")
            if passed:
                max_n_passed = n
            else:
                break
        self.beta_star  = beta_star
        if max_n_passed is None:
            print("Please decrease min_n")
        else:
            print(f"Q-Score: {max_n_passed}")
        
        if max_n_passed == self.max_n:
            print("Please increase max_n")


## Run on noisy simulator

In [9]:
device_name = "noiseless_sim"
submitter = CircuitSubmitter("qaoa-qscore", device_name)


In [10]:
benchmark = QScoreBenchmark(3, 4, depth, submitter=submitter)
benchmark.run()

  0%|          | 0/100 [00:00<?, ?it/s]

Q-Score benchmark with n=3 gives $\beta^*$=0.3885280461003305 > 0.2. Passed!


  0%|          | 0/100 [00:00<?, ?it/s]

Q-Score benchmark with n=4 gives $\beta^*$=0.4368471295646071 > 0.2. Passed!
Q-Score: 4
Please increase max_n
